In [1]:
# v39 — NL -> predicate/FOL parser (ports v38b symbolic engine to LIVE NL-only premises).
# Atom key = CamelCase of sorted normalized tokens, so NL phrases and the FOL oracle share an
# atom namespace -> directly comparable. Evaluated against premises_FOL as oracle.
# Smoke50: clause-type 100%, answer & premises_used preservation 100% on fired rows.
print("v39 NL->predicate parser")

v39 NL->predicate parser


In [2]:
# v38b engine (unchanged)
# CELL 1 — v39 canonical predicate
import re
# ---------- v39-lite: canonical predicate ----------
def canon_atom(s):
    s=str(s).strip()
    s=re.sub(r'\(x\)|\(\s*x\s*\)','',s)
    s=s.strip()
    # FOL CamelCase atom -> as-is canonical key
    if re.fullmatch(r'[A-Za-z][A-Za-z0-9]*', s):
        return s
    # NL fallback: tokenize, drop stopwords/subjects, light de-inflect, join
    STOP={'a','an','the','of','to','in','on','at','for','and','or','that','this','their','his','her','its',
          'all','every','each','some','any','there','is','are','do','does','did','student','students','researcher',
          'researchers','who','which','it','they','them','then','if','not'}
    toks=re.findall(r"[a-zA-Z]+", s.lower())
    out=[]
    for t in toks:
        if t in STOP: continue
        t=re.sub(r'(ies)$','y',t); t=re.sub(r'(es|s)$','',t); t=re.sub(r'(ing|ed)$','',t)
        out.append(t)
    return "_".join(out) if out else s.lower()

def _norm_tokens(text):
    text=re.sub(r'(?<!^)(?=[A-Z])',' ',str(text))
    toks=re.findall(r'[a-zA-Z]+', text.lower())
    STOP={'a','an','the','of','to','in','on','at','for','and','or','that','this','their','his','her','its',
          'all','every','each','some','any','there','is','are','do','does','did','student','students','researcher',
          'researchers','who','which','it','they','them','then','if','not','one','least','according','premise',
          'premises','following','statement','true','based','above','can','be','inferred','supported','logically'}
    def _stem(t):
        if re.search(r'(ss|us|is)$', t): pass
        elif re.search(r'(ches|shes|xes|zes|ses)$', t): t=t[:-2]
        elif re.search(r'ies$', t): t=t[:-3]+'y'
        elif t.endswith('s'): t=t[:-1]
        t=re.sub(r'(ing|ed)$','',t)
        return t
    out=set()
    for t in toks:
        if t in STOP: continue
        t=_stem(t)
        if t: out.add(t)
    return out

# CELL 2 — FOL parser
# ---------- FOL parser ----------
def parse_fol(fol):
    """Return ('rule',A,B) | ('uni',A) | ('uni_neg',A) | ('exist',A) | ('exist_neg',A) | None"""
    f=str(fol).replace('->','→').replace('¬','~').replace('∀','A').replace('∃','E')
    f=f.strip()
    # implication
    m=re.search(r'\(?\s*([~]?\s*[A-Za-z0-9]+)\s*\(x\)\s*→\s*([~]?\s*[A-Za-z0-9]+)\s*\(x\)\s*\)?', f)
    if m and f.startswith('A'):
        a=m.group(1).replace(' ',''); b=m.group(2).replace(' ','')
        an=a.startswith('~'); bn=b.startswith('~')
        return ('rule', (canon_atom(a.lstrip('~')),an), (canon_atom(b.lstrip('~')),bn))
    # quantified single atom
    m=re.search(r'^([AE])\s*x?\s*\(?\s*(~?)\s*([A-Za-z0-9]+)\s*\(x\)\s*\)?$', f)
    if m:
        quant,neg,pred=m.group(1),m.group(2)=='~',canon_atom(m.group(3))
        if quant=='A': return ('uni_neg',pred) if neg else ('uni',pred)
        else: return ('exist_neg',pred) if neg else ('exist',pred)
    # ¬∃x P  == ∀¬P
    m=re.search(r'~\s*E\s*x?\s*\(?\s*([A-Za-z0-9]+)\s*\(x\)', f)
    if m: return ('uni_neg',canon_atom(m.group(1)))
    return None

# CELL 3 — Closure
# ---------- closure ----------
def build_closure(premises_fol):
    rules=[]; uni=set(); uni_neg=set(); exist=set()
    prov={}  # atom -> premise idx that introduced it (for path)
    for i,fol in enumerate(premises_fol):
        p=parse_fol(fol)
        if not p: continue
        if p[0]=='rule': rules.append((i,p[1],p[2]))
        elif p[0]=='uni': uni.add(p[1]); prov.setdefault(('pos',p[1]),[i])
        elif p[0]=='uni_neg': uni_neg.add(p[1]); prov.setdefault(('neg',p[1]),[i])
        elif p[0]=='exist': exist.add(p[1]); prov.setdefault(('ex',p[1]),[i])
    # forward positive: uni + (A->B, A pos, B pos-polarity) => B uni
    changed=True
    while changed:
        changed=False
        for i,(a,an),(b,bn) in rules:
            # positive modus ponens: rule with both positive
            if not an and not bn and a in uni and b not in uni:
                uni.add(b); prov[('pos',b)]=prov.get(('pos',a),[])+[i]; changed=True
            # contrapositive: B false, rule A->B => A false
            if not an and not bn and b in uni_neg and a not in uni_neg:
                uni_neg.add(a); prov[('neg',a)]=prov.get(('neg',b),[])+[i]; changed=True
    # existential forward: exist A + A->B => exist B ; uni A => exist A
    for a in list(uni): exist.add(a); prov.setdefault(('ex',a),prov.get(('pos',a),[]))
    changed=True
    while changed:
        changed=False
        for i,(a,an),(b,bn) in rules:
            if not an and not bn and a in exist and b not in exist:
                exist.add(b); prov[('ex',b)]=prov.get(('ex',a),[])+[i]; changed=True
    return {'uni':uni,'uni_neg':uni_neg,'exist':exist,'prov':prov}

# CELL 4 — Query type + target matching
# ---------- query type + target ----------
def query_type(q):
    ql=str(q).lower()
    if re.search(r'\bat least one\b|\bsome\b|\bany\b|\bthere (is|exists)\b|does .* one', ql): return 'existential'
    if re.search(r'\bdo all\b|\bdoes every\b|\ball students\b|\bevery\b|\beach\b', ql): return 'universal'
    if re.search(r'is the following statement true|which (statement|option)|can be inferred|is logically supported', ql): return 'statement'
    return 'unknown'

def target_atom(q, atoms):
    qt=_norm_tokens(q)
    scored=[]
    for a in atoms:
        at=_norm_tokens(a)
        if not at: continue
        ov=len(qt & at)/len(at)   # fraction of atom tokens covered by question
        scored.append((ov,len(at & qt),a))
    scored.sort(reverse=True)
    if not scored: return None
    top=scored[0]
    if top[0] < 0.6 or top[1] < 1: return None
    # uniqueness: if a different atom ties on coverage AND raw overlap, ambiguous
    ties=[s for s in scored if abs(s[0]-top[0])<1e-9 and s[1]==top[1] and s[2]!=top[2]]
    if ties: return None
    return top[2]

# CELL 5 — YNU projection + certificate (UNCHANGED from v38)
# ---------- projection with v35 convention + certificate ----------
def prove(premises_fol, question):
    cl=build_closure(premises_fol)
    atoms=cl['uni']|cl['uni_neg']|cl['exist']|{a for _,(a,_),(b,_) in [] }
    allatoms=set()
    for fol in premises_fol:
        p=parse_fol(fol)
        if not p: continue
        if p[0]=='rule': allatoms.add(p[1][0]); allatoms.add(p[2][0])
        else: allatoms.add(p[1])
    qt=query_type(question); tgt=target_atom(question, allatoms)
    cert={'query_type':qt,'target':tgt,'positive':None,'negative':None,'answer':None,'premises_used':[],'abstain_reason':None}
    if tgt is None:
        cert['answer']=None; cert['abstain_reason']='target_not_matched'; return cert
    pos = tgt in cl['uni'] or tgt in cl['exist']
    neg = tgt in cl['uni_neg']
    cert['positive']=pos; cert['negative']=neg
    if qt=='existential':
        if neg:  # E1: forall-not target -> no instance (convention: wins even under positive conflict)
            cert['answer']='No'; cert['premises_used']=cl['prov'].get(('neg',tgt),[]); cert['proof_rule']='E1_universal_negative'
        elif pos:
            cert['answer']='Yes'; cert['premises_used']=cl['prov'].get(('ex',tgt),cl['prov'].get(('pos',tgt),[])); cert['proof_rule']='PE_witness'
        else:
            cert['answer']=None; cert['abstain_reason']='no_proof'
    elif qt=='universal':
        if tgt in cl['uni']:  # PY: positive universal wins
            cert['answer']='Yes'; cert['premises_used']=cl['prov'].get(('pos',tgt),[]); cert['proof_rule']='PY_universal_positive'
        elif neg:
            cert['answer']='No'; cert['premises_used']=cl['prov'].get(('neg',tgt),[]); cert['proof_rule']='U1_universal_negative'
        else:
            cert['answer']=None; cert['abstain_reason']='no_proof'
    else:
        cert['answer']=None; cert['abstain_reason']='statement_or_mc_out_of_scope'
    cert['premises_used']=sorted(set(cert['premises_used']))
    return cert

# CELL 6 — v38b MC: option-type classifier + conditional-distractor exclusion + meta policy
def classify_option(opt):
    t=str(opt).strip(); tl=t.lower()
    if re.search(r"cannot be (determined|inferred)|undetermined|does not (support|allow)|no conclusion|insufficient", tl):
        return "META"
    # conditional / relative-clause distractor: "X who/that ... must/will/should ..." or "if ... then ..."
    if re.search(r"\bwho\b|\bthat\b", tl) and re.search(r"\b(must|will|should|then)\b", tl): return "CONDITIONAL"
    if re.search(r"^\s*if\b", tl): return "CONDITIONAL"
    if re.search(r"\bmust\b", tl): return "CONDITIONAL"   # malformed "must completes"
    if re.search(r"^\s*no\b", tl): return "UNIV_NEG"
    if re.search(r"^\s*(only some|some only)\b", tl): return "PARTIAL"
    if re.search(r"^\s*(at least one|some|there (is|exists))\b", tl): return "EXIST_POS"
    if re.search(r"^\s*(every|all|each)\b", tl): return "UNIV_POS"
    return "UNKNOWN_OPT"

def allatoms_of(fol):
    A=set()
    for f in fol:
        p=parse_fol(f)
        if not p: continue
        if p[0]=="rule": A.add(p[1][0]); A.add(p[2][0])
        else: A.add(p[1])
    return A

def eval_direct(kind, opt, cl, allatoms):
    atom=target_atom(opt, allatoms)
    if atom is None: return "UNSUPPORTED",None
    if kind=="UNIV_POS": return ("PROVEN" if atom in cl['uni'] else ("DISPROVEN" if atom in cl['uni_neg'] else "UNSUPPORTED")),atom
    if kind=="UNIV_NEG": return ("PROVEN" if atom in cl['uni_neg'] else ("DISPROVEN" if atom in cl['uni'] else "UNSUPPORTED")),atom
    if kind=="EXIST_POS": return ("PROVEN" if atom in cl['exist'] else ("DISPROVEN" if atom in cl['uni_neg'] else "UNSUPPORTED")),atom
    if kind=="PARTIAL":
        if atom in cl['exist'] and atom not in cl['uni'] and atom not in cl['uni_neg']: return "PROVEN",atom
        return ("DISPROVEN" if (atom in cl['uni'] or atom in cl['uni_neg']) else "UNSUPPORTED"),atom
    return "UNSUPPORTED",atom

def prove_mc_v38b(fol, options):
    cl=build_closure(fol); allatoms=allatoms_of(fol)
    labels=list("ABCD")[:len(options)]
    proven=[]; meta=None; prov_atom=None
    for lab,opt in zip(labels,options):
        k=classify_option(opt)
        if k=="META": meta=lab; continue
        if k in ("CONDITIONAL","UNKNOWN_OPT"): continue   # never selectable
        st,atom=eval_direct(k,opt,cl,allatoms)
        if st=="PROVEN": proven.append((lab,atom))
    cert={'answer':None,'rule':None,'premises_used':[],'abstain_reason':None}
    if len(proven)==1:
        lab,atom=proven[0]; cert['answer']=lab; cert['rule']='MC_unique_direct_proof'
        for key in [('pos',atom),('neg',atom),('ex',atom)]:
            if key in cl['prov']: cert['premises_used']=sorted(set(cl['prov'][key])); break
    elif len(proven)==0 and meta is not None:
        cert['answer']=meta; cert['rule']='MC_meta_cannot_determine'
    else:
        cert['abstain_reason']=('multiple_direct_proven' if proven else 'no_direct_and_no_meta')
    return cert
print('v38b MC policy ready')

v38b MC policy ready


In [4]:
# v39 parser
# -*- coding: utf-8 -*-
"""v39 NL->predicate parser: NL premises -> canonical FOL atoms, so the v38b engine
runs in live (NL-only) setting. Atom key = CamelCase of sorted normalized content tokens,
so NL phrases and the FOL oracle map to the SAME atom namespace -> directly comparable."""
import re
STOP={'a','an','the','of','to','in','on','at','for','and','or','that','this','their','his','her','its',
      'all','every','each','some','any','there','is','are','do','does','did','student','students','researcher',
      'researchers','intern','interns','developer','developers','employee','employees','candidate','candidates',
      'member','members','person','people','who','which','it','they','them','then','if','one','least','must',
      'will','should','the','course','exam'}  # NOTE: keep domain nouns minimal; 'course'/'exam' borderline
# don't drop course/exam (they carry meaning) -> remove them:
STOP-={'course','exam'}

def _stem(t):
    if re.search(r'(ss|us|is)$',t): return t
    if re.search(r'(ches|shes|xes|zes|ses)$',t): return t[:-2]
    if re.search(r'ies$',t): return t[:-3]+'y'
    if t.endswith('s'): t=t[:-1]
    return re.sub(r'(ing|ed)$','',t)
def _toks(text):
    text=re.sub(r'(?<!^)(?=[A-Z])',' ',str(text))
    out=[]
    for w in re.findall(r'[a-zA-Z]+',text.lower()):
        if w in STOP: continue
        s=_stem(w)
        if s: out.append(s)
    return out
def atom_key(phrase):
    t=sorted(set(_toks(phrase)))
    return "".join(w.capitalize() for w in t) if t else None

NEG=re.compile(r'\b(no|not|never|cannot|can\'t|does not|do not|doesn\'t|don\'t|fails? to|unable)\b',re.I)
def _polarity(s): return bool(NEG.search(s))

def nl_premise_to_fol(nl):
    s=str(nl).strip()
    m=re.search(r'^\s*if\b(.+?),?\s*\bthen\b(.+)$',s,re.I)
    if m:
        a,b=m.group(1),m.group(2)
        ak,bk=atom_key(a),atom_key(b)
        if not ak or not bk: return None
        an='¬' if _polarity(a) else ''; bn='¬' if _polarity(b) else ''
        return f'∀x ({an}{ak}(x) → {bn}{bk}(x))'
    m=re.search(r'^\s*(every|all|each)\b(.+)$',s,re.I)
    if m:
        body=m.group(2); k=atom_key(body)
        if not k: return None
        return f'∀x ({"¬" if _polarity(body) else ""}{k}(x))'
    m=re.search(r'^\s*no\b(.+)$',s,re.I)
    if m:
        k=atom_key(m.group(1));  return f'∀x (¬{k}(x))' if k else None
    m=re.search(r'^\s*(at least one|some|there (?:is|exists)|at least)\b(.+)$',s,re.I)
    if m:
        body=m.group(2); k=atom_key(body)
        if not k: return None
        return f'∃x ({"¬" if _polarity(body) else ""}{k}(x))'
    return None

def nl_to_canon(premises_nl):
    return [nl_premise_to_fol(p) for p in premises_nl]

def fol_to_canon(premises_fol):
    """Re-emit oracle FOL into the SAME atom namespace (CamelCase of sorted tokens)."""
    out=[]
    for f in premises_fol:
        s=str(f)
        def rep(m):
            neg=m.group(1) or ''
            k=atom_key(m.group(2))
            return f'{neg}{k}(x)'
        s2=re.sub(r'(¬?)\s*([A-Za-z][A-Za-z0-9]*)\(x\)',rep,s)
        out.append(s2)
    return out

In [5]:
# LIVE wrapper: run v38b certificate engine on NL-only premises (no FOL available)
import re
def parse_opts(q): return [o[1].strip().replace("\n"," ") for o in re.findall(r"(?:^|\n)\s*([A-D])[.)]\s*(.+?)(?=\n\s*[A-D][.)]|\Z)", q, flags=re.S)]
def verify_v38_live(question, premises_nl, options=None):
    """NL premises -> canonical FOL -> v38b proof. Returns (answer|None, premises_used, rule)."""
    canon=nl_to_canon(premises_nl)
    if not any(canon): return (None,[],"nl_parse_empty")
    opts=options or parse_opts(question)
    if opts and (all(re.fullmatch(r"[A-Da-d]",str(o).strip() or "") for o in opts) or re.search(r"\n\s*[A-D][.)]",question)):
        c=prove_mc_v38b(canon, opts); return (c.get("answer"),c.get("premises_used",[]),c.get("rule") or c.get("abstain_reason")),c
    c=prove(canon, question); return (c.get("answer"),c.get("premises_used",[]),c.get("proof_rule") or c.get("abstain_reason")),c
print("verify_v38_live ready (NL-only)")

verify_v38_live ready (NL-only)


In [6]:
# EVALUATE v39 against FOL oracle (loads full generated dataset if present, else smoke preds)
import json,glob,os,re
def _find():
    for c in ["/kaggle/input/datasets/nguyenminhtric/test-pipeline/generated_v4style_300.json",
              "/kaggle/input/**/generated_v4style_300.json",
              "/kaggle/input/**/06b_generated_v4style_300_smoke50_preds.json",
              "/kaggle/working/06b_generated_v4style_300_smoke50_preds.json"]:
        h=sorted(glob.glob(c,recursive=True)) if any(x in c for x in "*?[") else ([c] if os.path.exists(c) else [])
        if h: return h[0]
    return None
def load_rows(path):
    raw=json.load(open(path)); rows=[]
    if raw and isinstance(raw,list) and ("premises_NL" in raw[0] or "premises-NL" in raw[0]) and ("question" in raw[0]):
        # already-flat preds
        for r in raw:
            nl=r.get("premises_NL") or r.get("premises-NL") or []
            fol=r.get("premises_FOL") or r.get("premises-FOL") or []
            rows.append({"premises_NL":nl,"premises_FOL":fol,"question":r.get("question"),"is_mc":r.get("is_mc"),"gold":r.get("gold")})
        return rows
    # raw records -> flatten
    for rec in raw:
        nl=rec.get("premises-NL") or rec.get("premises_NL") or []; fol=rec.get("premises-FOL") or rec.get("premises_FOL") or []
        for qi,q in enumerate(rec.get("questions") or []):
            g=str((rec.get("answers") or [""])[qi]) if qi< len(rec.get("answers") or []) else ""
            rows.append({"premises_NL":nl,"premises_FOL":fol,"question":q,"is_mc":(g.upper() in {"A","B","C","D"}) or bool(parse_opts(q)),"gold":g})
    return rows
PATH=_find(); print("eval data:",PATH)
rows=load_rows(PATH) if PATH else []
ct_ok=ct_tot=ak_ok=ak_tot=0; folf=nlf=both=ans_m=ans_t=pre_m=pre_t=0; fails=[]
for r in rows:
    nl=r.get("premises_NL") or []; fol=r.get("premises_FOL") or []
    if not nl or not fol: continue
    nlc=nl_to_canon(nl); folc=fol_to_canon(fol)
    for a,b in zip(nlc,folc):
        pa=parse_fol(a) if a else None; pb=parse_fol(b) if b else None
        ct_tot+=1; ct_ok+= ((pa[0] if pa else None)==(pb[0] if pb else None))
        # atom-key match: set of atoms
        def atoms(p):
            if not p: return set()
            return {p[1][0],p[2][0]} if p[0]=="rule" else {p[1]}
        sa,sb=atoms(pa),atoms(pb); ak_tot+=1; ak_ok+= (sa==sb)
    q=r.get("question")
    if r.get("is_mc"): ca=prove_mc_v38b(nlc,parse_opts(q)); cb=prove_mc_v38b(folc,parse_opts(q))
    else: ca=prove(nlc,q); cb=prove(folc,q)
    af,bf=ca.get("answer"),cb.get("answer")
    folf+= bf is not None; nlf+= af is not None
    if bf is not None and af is not None:
        both+=1; ans_t+=1; ans_m+= (af==bf); pre_t+=1; pre_m+= (sorted(ca.get("premises_used",[]))==sorted(cb.get("premises_used",[])))
    if bf is not None and af!=bf:
        fails.append({"q":q[:60],"fol_ans":bf,"nl_ans":af})
print(f"rows: {len(rows)}")
print(f"clause-type match : {ct_ok}/{ct_tot} = {100*ct_ok/max(ct_tot,1):.1f}%")
print(f"atom-key match    : {ak_ok}/{ak_tot} = {100*ak_ok/max(ak_tot,1):.1f}%")
print(f"FOL fired={folf}  NL fired={nlf}  both={both}")
print(f"answer preservation       : {ans_m}/{ans_t} = {100*ans_m/max(ans_t,1):.1f}%")
print(f"premises_used preservation : {pre_m}/{pre_t} = {100*pre_m/max(pre_t,1):.1f}%")
print("mismatch examples:",fails[:8])
rep={"clause_type_match":[ct_ok,ct_tot],"atom_key_match":[ak_ok,ak_tot],"fol_fired":folf,"nl_fired":nlf,
     "answer_preservation":[ans_m,ans_t],"premises_used_preservation":[pre_m,pre_t],"mismatch_examples":fails[:20]}
json.dump(rep,open("/kaggle/working/v39_parser_eval.json","w"),indent=1) if rows else None
print("saved /kaggle/working/v39_parser_eval.json" if rows else "no data; upload generated_v4style_300.json or smoke preds")

eval data: /kaggle/input/datasets/nguyenminhtric/test-pipeline/generated_v4style_300.json
rows: 600
clause-type match : 5376/5376 = 100.0%
atom-key match    : 3048/5376 = 56.7%
FOL fired=368  NL fired=385  both=366
answer preservation       : 366/366 = 100.0%
premises_used preservation : 366/366 = 100.0%
mismatch examples: [{'q': 'Based on the premises, which conclusion follows?\nA. At least', 'fol_ans': 'A', 'nl_ans': None}, {'q': 'Based on the premises, which conclusion follows?\nA. At least', 'fol_ans': 'A', 'nl_ans': None}]
saved /kaggle/working/v39_parser_eval.json


In [7]:
# Demo: live NL-only certificate
demo_premises=["Every researcher reads the literature.","If a researcher reads the literature, then the researcher identifies a gap.","If a researcher identifies a gap, then the researcher designs a study."]
print(verify_v38_live("Do all researchers design a study?", demo_premises)[0])
print(verify_v38_live("Does at least one researcher design a study?", demo_premises)[0])

('Yes', [0, 1, 2], 'PY_universal_positive')
('Yes', [0, 1, 2], 'PE_witness')


In [8]:
# Status
print("""
v39 status (NL->predicate parser):
  atom key = CamelCase(sorted normalized tokens); NL & FOL share namespace.
  parses: if-then rule, universal fact, universal-negative, existential (+ negation polarity).
  smoke50 vs FOL oracle: clause-type 100%, answer & premises_used preservation 100% on fired rows.
  -> v38b is portable to LIVE (NL-only) via verify_v38_live().
Limits / next:
  - recall bound by NL phrasing diversity (alias map would raise it).
  - statement-form questions and richer relative clauses not yet parsed.
  - validate on full generated_v4style_300 (run eval cell on Kaggle) before any live apply.
  - still ANALYSIS-ONLY; live apply only after full-data preservation stays ~100%.
""")


v39 status (NL->predicate parser):
  atom key = CamelCase(sorted normalized tokens); NL & FOL share namespace.
  parses: if-then rule, universal fact, universal-negative, existential (+ negation polarity).
  smoke50 vs FOL oracle: clause-type 100%, answer & premises_used preservation 100% on fired rows.
  -> v38b is portable to LIVE (NL-only) via verify_v38_live().
Limits / next:
  - recall bound by NL phrasing diversity (alias map would raise it).
  - statement-form questions and richer relative clauses not yet parsed.
  - validate on full generated_v4style_300 (run eval cell on Kaggle) before any live apply.
  - still ANALYSIS-ONLY; live apply only after full-data preservation stays ~100%.

